In [2]:
import os, zipfile, requests

urls = {
    "native": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/native_script_train_valid_data.zip",
    "roman": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/roman_script_train_valid_data.zip",
    "native-roman": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/parallel_romanized_train_data.zip",
    "abhijnaanam": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/bhasha-abhijnaanam_test_set.zip"
}

os.makedirs("ba_training", exist_ok=True)

for name, url in urls.items():
    zip_path = f"ba_training/{name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {name} dataset...")
        r = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(f"ba_training/")
        os.remove(zip_path)
        print("Zip file removed.")
os.listdir("ba_training/")

Zip file removed.
Zip file removed.
Zip file removed.
Zip file removed.


['parallel_romanized_train_data.json',
 'Native_script_data',
 'Roman_script_data',
 'bhasha-abhijnaanam.json']

In [3]:
import pandas as pd

def load_and_sample(path, frac=1, seed=42):
    """Load the text file, split label and text, and sample a portion of data."""
    data = []

    # Open the file and read line by line
    with open(path, 'r') as f:
        for line in f:
            # Split line into label and text
            parts = line.strip().split(maxsplit=1)  # Only split at the first space
            if len(parts) == 2:
                label, text = parts
                label = label.replace("__label__","")
                data.append([label, text])  # Append label and text as a list

    # Create DataFrame
    df = pd.DataFrame(data, columns=["label", "text"])

    # Sample a fraction of the data
    # df = df.sample(frac=frac, random_state=seed)

    return df

roman_train = load_and_sample("ba_training/Roman_script_data/train_combine.txt")
roman_valid = load_and_sample("ba_training/Roman_script_data/valid_combine.txt")

native_train = load_and_sample("ba_training/Native_script_data/train_combine.txt")
native_valid = load_and_sample("ba_training/Native_script_data/valid_combine.txt")

In [4]:
import json
import pandas as pd

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)   # one JSON object
    data = obj["data"]

    # build DataFrame
    df = pd.DataFrame([{
        "id": d["unique_identifier"],
        "native": d["native sentence"],
        "roman": d["romanized sentence"],
        "label": d["language"],
        "script": d["script"],
        "source": d["source"]
    } for d in data])

    return df

# usage
# benchmark = load_json("ba_training/bhasha-abhijnaanam.json")
roman_native = load_json("ba_training/parallel_romanized_train_data.json")

In [5]:
# Keep Manipuri as it is, undersample others to 100k each
sampled_dfs = []

for lang, group in roman_native.groupby("label"):
    if lang == "Manipuri":
        sampled = group  # keep all Manipuri samples
    else:
        sampled = group.sample(n=100_000, random_state=42)
    sampled_dfs.append(sampled)

# Combine into a single DataFrame
native_roman_balanced = pd.concat(sampled_dfs, ignore_index=True)

# Print summary info
print("Samples per language (after undersampling others and keeping Manipuri as is):")
print(native_roman_balanced["label"].value_counts())

# Calculate size in bytes → MB / GB
df_size_bytes = native_roman_balanced.memory_usage(deep=True).sum()
df_size_mb = df_size_bytes / (1024 ** 2)
df_size_gb = df_size_bytes / (1024 ** 3)

print(f"\nTotal samples: {len(native_roman_balanced):,}")
print(f"Languages included: {native_roman_balanced['label'].nunique()}")
print(f"DataFrame size: {df_size_mb:.2f} MB ({df_size_gb:.3f} GB)")


Samples per language (after undersampling others and keeping Manipuri as is):
label
Assamese     100000
Bangla       100000
Bodo         100000
Gujarati     100000
Hindi        100000
Kannada      100000
Kashmiri     100000
Konkani      100000
Maithili     100000
Malayalam    100000
Marathi      100000
Nepali       100000
Sindhi       100000
Oriya        100000
Punjabi      100000
Sanskrit     100000
Telugu       100000
Tamil        100000
Urdu         100000
Manipuri      30528
Name: count, dtype: int64

Total samples: 1,930,528
Languages included: 20
DataFrame size: 1230.61 MB (1.202 GB)


In [6]:
from datasets import load_dataset
import random

langs = ["hi", "bn", "ta", "ml", "te", "gu", "mr", "pa", "or", "kn", "as"]
english_sentences = []

for lang in langs:
    try:
        print(f"Loading {lang} split from Samanantar...")
        ds = load_dataset("ai4bharat/samanantar", lang, split="train[:200000]")

        # Filter sentences that are strings and have between 10 and 50 words
        filtered = [
            row["src"] for row in ds
            if isinstance(row["src"], str)
            and 10 <= len(row["src"].split()) <= 50
        ]

        english_sentences.extend(filtered)
        print(f"Loaded {len(filtered):,} valid English sentences for {lang}")

    except Exception as e:
        print(f"Skipping {lang} due to error: {e}")

print(f"Total English sentences collected: {len(english_sentences):,}")

# Sanity check
if not english_sentences:
    raise RuntimeError("No English sentences collected! Check dataset structure or filtering criteria.")

# Randomly assign one English sentence to each row
native_roman_balanced["english_random_sentence"] = random.choices(
    english_sentences, k=len(native_roman_balanced)
)

print("Added 'english_random_sentence' column from Samanantar dataset.")

Loading hi split from Samanantar...
Loaded 129,740 valid English sentences for hi
Loading bn split from Samanantar...
Loaded 81,276 valid English sentences for bn
Loading ta split from Samanantar...
Loaded 81,614 valid English sentences for ta
Loading ml split from Samanantar...
Loaded 75,598 valid English sentences for ml
Loading te split from Samanantar...
Loaded 68,754 valid English sentences for te
Loading gu split from Samanantar...
Loaded 78,150 valid English sentences for gu
Loading mr split from Samanantar...
Loaded 85,900 valid English sentences for mr
Loading pa split from Samanantar...
Loaded 105,667 valid English sentences for pa
Loading or split from Samanantar...
Loaded 86,231 valid English sentences for or
Loading kn split from Samanantar...
Loaded 64,600 valid English sentences for kn
Loading as split from Samanantar...
Loaded 66,152 valid English sentences for as
Total English sentences collected: 923,682
Added 'english_random_sentence' column from Samanantar dataset.


In [7]:
# Randomly assign one English sentence to each row
roman_native["english_random_sentence"] = random.choices(
    english_sentences, k=len(roman_native)
)
print("Added 'english_random_sentence' column from Samanantar dataset.")

Added 'english_random_sentence' column from Samanantar dataset.


In [16]:
native_roman_balanced.sample()['english_random_sentence'].values

array(["Gandhi had on Monday attacked Prime Minister Narendra Modi over corruption and asked how all thieves have 'Modi' as the common surname as he referred to fugitive businessman Nirav Modi and former IPL chairman Lalit Modi"],
      dtype=object)

In [9]:
native_roman_balanced.to_csv("ba_training_triplets/abhijnaanam_balanced_triplets.csv", index=False)
roman_native.to_csv("ba_training_triplets/abhijnaanam_triplets.csv", index=False)

In [14]:
len(roman_native), len(native_train) + len(native_valid), len(roman_train) + len(roman_valid)

(4396964, 2710951, 2253097)

In [12]:
roman_native['label'].value_counts()

label
Telugu       299033
Gujarati     298984
Bangla       298926
Malayalam    298812
Marathi      298795
Hindi        298612
Oriya        296294
Tamil        292926
Kannada      289949
Assamese     279066
Punjabi      235848
Nepali       234596
Sanskrit     201462
Maithili     156921
Sindhi       150751
Bodo         114102
Konkani      110001
Urdu         105704
Kashmiri     105654
Manipuri      30528
Name: count, dtype: int64

In [18]:
native = pd.concat([native_train, native_valid], axis=0, ignore_index=True)
native['label'].value_counts()

label
Nepali           117051
Urdu             111658
Bodo             108740
Telugu           105997
Tamil            105997
Kannada          105997
Punjabi          105997
Gujarati         105997
Marathi          105997
Hindi            105997
Malayalam        105997
Bangla           105997
Sindhi           105996
Sanskrit         103812
Maithili         101774
Kashmiri_Arab    101497
Oriya            100997
Manipuri_Beng    100997
English          100997
Assamese         100997
Kashmiri_Deva    100997
Manipuri_Mei     100500
Konkani          100500
Santali          100345
Dogri            100120
Other            100000
Name: count, dtype: int64

In [19]:
roman = pd.concat([roman_train, roman_valid], axis=0, ignore_index=True)
roman['label'].value_counts()

label
Nepali          116520
Bodo            109239
Urdu            106102
Sanskrit        103315
Maithili        101277
Manipuri_Mei    101000
Konkani         101000
Gujarati        101000
Hindi           101000
Punjabi         101000
English         101000
Assamese        101000
Kannada         101000
Tamil           101000
Telugu          101000
Other           101000
Malayalam       101000
Marathi         101000
Oriya           101000
Sindhi          101000
Bangla          101000
Kashmiri        100644
Name: count, dtype: int64

In [20]:
native_roman_balanced.to_csv("abhijnaanam_balanced_triplets.csv", index=False)
roman_native.to_csv("abhijnaanam_triplets.csv", index=False)
native.to_csv("abhijnaanam_native.csv", index=False)
roman.to_csv("abhijnaanam_roman.csv", index=False)

In [21]:
benchmark = load_json("ba_training/bhasha-abhijnaanam.json")
benchmark.head()

,id,native,roman,label,script,source
0,as_2,অৱশ্যে লাচিত সংঘৰ সদস্য তথা বিষয়ববীয়াই এই পৰিস...,Owoshye lachit sanghar sadasya totha bixoybobi...,Assamese,Bengali,Manually-Collected
1,as_3,শোকস্তব্ধ হৈ পৰে সমগ্ৰ অঞ্চলটো৷ বি বি এছ বি হা...,Shukostobddho hoi pore samogro anchaltu. BBSB ...,Assamese,Bengali,Manually-Collected
2,as_4,তেতিয়া ভাবিব পাৰি৷ ডলীয়ে ৰাজশ্ৰী হৰিণীক চকু টি...,Tetiya bhabibo pari. Dollye Rajashri Horinik c...,Assamese,Bengali,Manually-Collected
3,as_5,৫৬ জন যাত্ৰী কঢ়িয়াই অনা বাছখন নদীত বাগৰি পৰাৰ ...,56 jon jatri kohiyai ana buskhon nodit bagori ...,Assamese,Bengali,Manually-Collected
4,as_6,আমেৰিকাৰ ৰাষ্ট্ৰপতি ডোনাল্ড ট্ৰাম্পে শুকুৰবাৰে...,Americar rastrapoti Donald Trumpe sukurbare ot...,Assamese,Bengali,Manually-Collected


In [22]:
benchmark.label.value_counts()

label
Urdu         6907
Sindhi       5901
Kannada      5861
Tamil        5814
Gujarati     5801
Punjabi      5794
Telugu       5766
Malayalam    5639
Marathi      5629
Hindi        5628
Bangla       5612
Kashmiri     3530
Sanskrit     2527
Maithili     2515
Manipuri     2514
Nepali       2514
Santali      2512
Assamese     1524
Oriya        1524
Bodo         1502
Konkani      1500
Dogri        1500
Name: count, dtype: int64

In [26]:
benchmark.label.nunique()

22

In [27]:
training_data = pd.read_csv('phase2.csv')
training_data.head()

,text,label
0,উল্লেখ্য যে যোৱা ডিচেম্বৰত নতুন কৃষি আইনক কেন্...,Assamese
1,আয়োগক মুকলিভাৱে সমৰ্থন কৰা মাত্ৰ কেইজনমান ৰিপ...,Assamese
2,বাহ্যিক বস্তু সাধাৰণতে চকুৰ ওপৰ বা তলপতাৰ তলিত...,Assamese
3,"তাৰোপৰি কেঁচা জলকীয়া , অমিতা , বিলাহী , অংকুৰ...",Assamese
4,এনে ৰোগত ৰোগীজনে বাস্তৱৰ লগত সম্পৰ্ক হেৰুৱাই ন...,Assamese


In [28]:
set(training_data.label.unique()) - set(benchmark.label.unique()), set(benchmark.label.unique()) - set(training_data.label.unique())

(set(), set())

In [29]:
benchmark.to_csv('bhasha-abhijnaanam.csv', index=False)